## Phân tích liệu CO₂ (OCO) với vùng trồng lúa bằng phương pháp vùng đệm quanh từng điểm CO₂ để ước tính tỷ lệ vùng trồng lúa trong phạm vi đó. Lọc dữ liệu theo ngưỡng tỷ lệ vùng trồng lúa

In [ ]:
import pandas as pd
import geopandas
from shapely.geometry import Point
import rasterio
from rasterio.mask import mask
import numpy as np
import os
import glob
import re

In [ ]:
def approximate_co2_rice_linkage(csv_path, raster_path, output_csv_path,
                                 buffer_radius_meters,
                                 rice_proportion_threshold, 
                                 lon_col='longitude', lat_col='latitude',
                                 co2_points_crs_epsg=4326, 
                                 quality_flag_col=None,
                                 valid_quality_flags=None):
    co2_df = pd.read_csv(csv_path)

    if quality_flag_col and valid_quality_flags is not None:
        original_rows = len(co2_df)
        co2_df = co2_df[co2_df[quality_flag_col].isin(valid_quality_flags)].copy()

    with rasterio.open(raster_path) as src_rice_map:
        rice_map_crs = src_rice_map.crs
        rice_map_nodata = src_rice_map.nodata
        print(f"  CRS bản đồ lúa: {rice_map_crs}")
        print(f"  NoData bản đồ lúa: {rice_map_nodata}")

        geometry = [Point(xy) for xy in zip(co2_df[lon_col], co2_df[lat_col])]
        co2_gdf = geopandas.GeoDataFrame(co2_df, geometry=geometry, crs=f"EPSG:{co2_points_crs_epsg}")
        if co2_gdf.crs != rice_map_crs:
            print(f"  Chiếu lại điểm CO2 từ {co2_gdf.crs} sang {rice_map_crs}...")
            co2_gdf = co2_gdf.to_crs(rice_map_crs)

        co2_gdf['buffer_geometry'] = co2_gdf.geometry.buffer(buffer_radius_meters)

        rice_proportions = []
        is_rice_influenced = []

        for index, row in co2_gdf.iterrows():
            buffer_geom = row['buffer_geometry']
                
            out_image, out_transform = mask(dataset=src_rice_map,
                                            shapes=[buffer_geom],
                                            crop=True,
                                            nodata=rice_map_nodata if rice_map_nodata is not None else -9999,  # Giá trị để fill vùng ngoài mask nếu crop=False
                                            filled=True)

            rice_pixels_in_buffer = out_image[0]
            if rice_map_nodata is not None:
                valid_pixels = rice_pixels_in_buffer[rice_pixels_in_buffer != rice_map_nodata]
            else: 
                valid_pixels = rice_pixels_in_buffer.flatten()

            if valid_pixels.size == 0: 
                rice_proportions.append(0.0)
                is_rice_influenced.append(False)
                continue

            # Đếm số pixel lúa (giá trị = 1)
            num_rice_pixels = np.sum(valid_pixels == 1)
            proportion = num_rice_pixels / valid_pixels.size
            rice_proportions.append(proportion)
            is_rice_influenced.append(proportion >= rice_proportion_threshold)

        co2_gdf['rice_proportion_in_buffer'] = rice_proportions
        co2_gdf['is_rice_influenced'] = is_rice_influenced
        output_df = pd.DataFrame(co2_gdf.drop(columns=['geometry', 'buffer_geometry']))

    if 'is_rice_influenced' in output_df.columns:
        influenced_count = output_df['is_rice_influenced'].sum() 
        print(f"Số điểm CO2 được xác định là ảnh hưởng bởi lúa (tỷ lệ >= {rice_proportion_threshold*100}%): {influenced_count}")
        filtered_output_df = output_df[output_df['is_rice_influenced'] == True]
        filtered_output_df.to_csv(output_csv_path, index=False, float_format='%.6f')

In [ ]:
def process_single_csv_file():
    # File CSV đầu vào (đã gộp tất cả dữ liệu từng năm)
    csv_file_path = r"E:\DownloadData\co2_nasa\data_processed\merge_oco2_oco3\oco_merged_2024.csv"

    # Thư mục đầu ra (chứa kết quả lọc dữ liệu trong vùng trồng lúa)
    filter_in_rice_folder = r"E:\DownloadData\co2_nasa\data_processed\oco_rice_buffer"
    os.makedirs(filter_in_rice_folder, exist_ok=True)

    # Đường dẫn bản đồ vùng trồng lúa (đã chiếu UTM)
    raster_rice_map_projected = r"E:\RanhGioi\LandCoverMap\lc_vn_250m_utm_rice_binary.tif"

    # Tên file đầu ra
    output_csv_full_path = os.path.join(filter_in_rice_folder, "oco_rice_buffer_2024.csv")

    # Thông số
    buffer_radius = 1000   # bán kính vùng đệm (mét)
    rice_threshold = 0.3   # ngưỡng tỉ lệ diện tích lúa trong vùng đệm

    # Gọi hàm xử lý
    approximate_co2_rice_linkage(
        csv_path=csv_file_path,
        raster_path=raster_rice_map_projected,
        output_csv_path=output_csv_full_path,
        buffer_radius_meters=buffer_radius,
        rice_proportion_threshold=rice_threshold,
        lon_col='longitude',
        lat_col='latitude',
        co2_points_crs_epsg=4326,
        quality_flag_col=None,
        valid_quality_flags=None
    )

    print(f"✅ Đã xử lý xong file: {os.path.basename(csv_file_path)}")
    print(f"➡️ Kết quả được lưu tại: {output_csv_full_path}")

In [ ]:
if __name__ == "__main__":
    process_single_csv_file()